In [25]:
import os
import time
import re

import pandas as pd
import requests
import urllib.parse
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

from Constants import Constants as const

In [21]:
# === 1. 初始化 Selenium 浏览器 ===
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager(driver_version="138").install()), options=options)

# 用 JS 覆盖 navigator.webdriver
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
  "source": """
    Object.defineProperty(navigator, 'webdriver', {
      get: () => undefined
    })
  """
})

{'identifier': '2'}

In [6]:
gvkey_glassdoor_url = pd.read_excel(os.path.join(const.TEMP_PATH, '20250625_ue_with_glassdoor_web_fill_miss.xlsx'))

In [22]:
driver.get('https://www.glassdoor.com/index.htm')

In [18]:
driver.find_element(by=By.ID, value='inlineUserEmail').send_keys('wangyouan@gmail.com')
driver.find_element(by=By.XPATH, value='//*[@id="InlineLoginModule"]/div/div[1]/div/div/div/div/form/div[2]/div/button').click()


In [19]:
driver.find_element(by=By.ID, value='inlineUserPassword').send_keys('4Pa5EjPghdTV-c5')
driver.find_element(by=By.XPATH, value='//*[@id="InlineLoginModule"]/div/div[1]/div/div/div/div/form/div[2]/div/button').click()

In [23]:
driver.close()

In [26]:
def slugify(s):
    # Glassdoor 评论页中的 shortName 通常无需处理，但保险起见进行标准清洗
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'\s+', '-', s.strip())
    return s

def get_glassdoor_reviews_url(company_name):
    base_api = "https://www.glassdoor.com/autocomplete/employers?term="
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        query = urllib.parse.quote(company_name)
        res = requests.get(base_api + query, headers=headers)
        if res.status_code == 200:
            data = res.json()
            if data:
                item = data[0]
                cid = item['id']
                short = slugify(item['shortName']) if 'shortName' in item else slugify(item['name'])
                return f"https://www.glassdoor.com/Reviews/{short}-Reviews-E{cid}.htm"
    except Exception as e:
        print(f"[Error] {company_name}: {e}")

    return None


In [28]:
def construct_review_url(company_name, company_id):
    short = slugify(company_name)
    return f"https://www.glassdoor.com/Reviews/{short}-Reviews-E{company_id}.htm"

In [29]:
print(construct_review_url('K-V Pharmaceutical', '2500'))

https://www.glassdoor.com/Reviews/K-V-Pharmaceutical-Reviews-E2500.htm


In [33]:
print('\n'.join([f'"{i}",' for i in '''Metro Inc
OptiCare Health Systems Inc.
Orthovita Inc
POSCO Holdings Inc
Trimac Transportation Ltd
Medco Health Solutions Inc.
Ramp Corp
Penske Truck Leasing Co LP
Weyerhaeuser Company Ltd
EPR Properties
BlueLinx Holdings Inc
Hyco International Inc
Johns Manville Corp
Extendicare Inc
Sevenson Environmental Services Inc
Comcast Cable Communications Inc.
Appalachian Power
Ryerson Tull Inc-Old
KT Corporation
West Marine Inc
Swift Transportation Co
Standard Register Co (The)
Public Service Company of New Hampshire
Regional Brands Inc
Connecticut Light & Power Co
Sure Energy Inc
Loomis Fargo & Co
Garda World Security Corp
Petro USA Inc
Gas Natural Inc
Verizon Inc/PA
Consolidated Container Co LLC
Vanguard Car Rental Group Inc
Tube City IMS Corp
Data Systems Inc
G&K Services Inc
West Energy Ltd
Noble Environmental Power Inc
Chrysler Group LLC
Bea Systems Inc
RAE Systems Inc.
America Service Group Inc
Black Box Corp
D-Box Technologies Inc
Wackenhut Corp
IFCI International Corp
Albertsons Cos Inc
Industrial Services of America Inc
Fresenius Medical Care AG
Pure World Inc
Certainteed Corp
Pennsylvania Power Co
Superior Bancorp
Stratus Services Group Inc
Renal Care Group Inc.
Central Freight Lines Inc
Car Inc
Constar International Inc
Cenveo Inc.
American Medical Security Group Inc
Stratford American Corp
Pathfinder Bancorp Inc
Titan Technologies Inc
Bunzl PLC
Boardwalk Pipelines LLC
LMI Aerospace Inc
MeadWestvaco Corp
Greyhound Lines Inc.
B Communications Ltd
Transportation Component Inc
Cooper Industries Plc
Ohio Power
Wheeling Island Gaming Inc
Dynamic Sciences International
Cushman & Wakefield Ltd
Domino's Inc.
Ferrellgas Partners LP
Arizona Public Service Co
WJ Communications Inc
DIRECTV Holdings LLC
L'Air Liquide SA
CEVA Logistics Inc
Rhodia
Spartech Corp
Tasty Baking Co
Dr. Pepper Bottling Holdings
ABB Ltd
ASI Liquidating Corp
Total Logistics Inc
Sodexho Marriott Services Inc
Community Care Services Inc
National Grid plc
Lafarge North America Inc.
Indiana Michigan Power Co
Energy East Corp.
Omega Protein Corp
Kentucky Power
Remington Arms Co Inc
Rand Logistics Inc
First Aviation Services Inc
BTU International Inc
AG Growth International Inc
Sunoco Logistics Partners LP
Alion Science and Technology Corp
Coleman Cable Inc
Columbus Southern Power Corp
Monongahela Power
Park Aerospace Corp
Maverick Minerals Corp
Commonwealth Edison Co
TFI International Inc
Capital Power Corp
Ohio Edison Co
Sierra Pacific Power Co
Strateco Resources Inc
CompX International Inc.
Inventure Foods Inc
Aquila Resources Inc
Curis Resources Ltd
Regal Entertainment Group
Univar Corp
Metropolitan Edison
Ralcorp Holdings Inc.
Verus International Inc
Pittston Co-Consolidated
Puget Sound Energy Inc
Axalta Coating Systems Ltd
AEP Industries Inc
Southern Power Co
Tecogen Inc
Central Maine Power Co
NES Rentals Holdings Inc
Pivot Technology Solutions Inc
Allegheny Energy Supply Co LLC
Care Capital Properties Inc
Refco Inc
Community Medical Transport
Ensign Energy Services Inc
Telecomunicaciones de Puerto Rico Inc.
Sense Technologies Inc
Dextera Surgical Inc
Skyline Corp
Northwest Pipe Co
Canon Inc
Colt Resources Inc
Cadus Corporation
Associated Materials LLC
SIFCO Industries Inc.
Superior Plus Corp
ICTS International NV
Progressive Waste Solutions Ltd
Exa Corp
Stantec Inc
Texas Mineral Resources Corp
SLR Investment Corp
Florida Power & Light Co
Hawaiian Electric Co
Danaos Corporation
DS Healthcare Group Inc
Opta Minerals Inc
Parks America Inc
Central Hudson Gas & Electric
Westar Energy Inc
International Game Technology PLC
KapStone Paper & Packaging Corp
Fort Dearborn Income Securities Inc.
Tampa Electric Co
Ore Holdings Inc
NBCUniversal Media LLC
Nuverra Environmental Solutions Inc
AdvancePierre Foods Holdings Inc
CSRA Inc
Lifetime Brands Inc
San Diego Gas & Electric Co
Fairmount Santrol Holdings Inc
BPO Management Services Inc
CWC Energy Services Corp
Jorgensen (Earle M.) Co
Eastern Co (The)
Anheuser-Busch InBev SA/NV
Baltimore Gas & Electric Co
Everest Healthcare Services Corp
Air Industries Group Inc
Command Security Corp
Atlantic American Corp
Atlantic City Electric Co
Genesis Healthcare Inc
Kentucky Utilities Co
Boyd Group Services Inc
Consumers Energy Co
SI Financial Group Inc
Land O'Lakes Inc.
Central Steel & Wire Co
GFL Environmental Inc
South Carolina Elec & Gas Co
Blue Apron Holdings Inc
Intercontinental Hotels Group PLC
Highcliff Metals Corp
United Security Bancshares
PECO Energy Co
CH2M HILL Companies Ltd.
Komatsu Ltd
People Corp
Wynn Las Vegas LLC
Duke Energy Carolinas LLC
UGI Utilities Inc
Armstrong Flooring Inc
Sprague Resources LP
Spirent Communications
Mercury Air Group Inc.
Unique Fabricating Inc
Agiliti Inc
KWG Resources Inc
United States Postal Service
Green Thumb Industries Inc
ALR Technologies SG LTD
International Baler Corp
Professional Transportation Group Inc
Max Sound Corp
Wisconsin Public Service Corp
Pason Systems Inc
North Shore Gas Co
Strattec Security Corp
Forterra Inc
Sterigenics International Inc
Carrier Access Corp
7-Eleven Inc
Koppers Inc.
Southwestern Electric Power Co
Oilgear Company (The)
TAT Technologies Ltd
Georgia Power Co
K-Sea Transportation Partners LP
Atlas Corp
Thomson Reuters Corp
Comprehensive Care Corp
RS Technologies Inc
Multiband Corp
Mitec Technologies Inc
China Natural Resources Inc
Dollar Thrifty Automotive Group Inc.
Lafarge SA
Centerline Holding Co
Teva Pharmaceutical Industries Ltd
Oplink Communications Inc
Gateway Energy Corp
Eagle Energy Inc
NSTAR Electric Co
Zale Corp
West Corp
Crude Carriers Corp
CPS Technologies Corp
Aecon Group Inc
Fujitsu Ltd
Silvan Industries Inc
Celera Corp
AmerenEnergy Generating Co'''.split('\n')]))

"Metro Inc",
"OptiCare Health Systems Inc.",
"Orthovita Inc",
"POSCO Holdings Inc",
"Trimac Transportation Ltd",
"Medco Health Solutions Inc.",
"Ramp Corp",
"Penske Truck Leasing Co LP",
"Weyerhaeuser Company Ltd",
"EPR Properties",
"BlueLinx Holdings Inc",
"Hyco International Inc",
"Johns Manville Corp",
"Extendicare Inc",
"Sevenson Environmental Services Inc",
"Comcast Cable Communications Inc.",
"Appalachian Power",
"Ryerson Tull Inc-Old",
"KT Corporation",
"West Marine Inc",
"Swift Transportation Co",
"Standard Register Co (The)",
"Public Service Company of New Hampshire",
"Regional Brands Inc",
"Connecticut Light & Power Co",
"Sure Energy Inc",
"Loomis Fargo & Co",
"Garda World Security Corp",
"Petro USA Inc",
"Gas Natural Inc",
"Verizon Inc/PA",
"Consolidated Container Co LLC",
"Vanguard Car Rental Group Inc",
"Tube City IMS Corp",
"Data Systems Inc",
"G&K Services Inc",
"West Energy Ltd",
"Noble Environmental Power Inc",
"Chrysler Group LLC",
"Bea Systems Inc",
"RAE Systems Inc.